> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataProcessing/blob/main/notebooks/04b_Many_Counties.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Many Counties: Automating a Workflow

**Series:** Automating Repeated Steps in Python (notebook 2 of 2)
**Time:** about 35 minutes

**In this notebook you will:**
- Package the Census cleaning steps from notebook 01a as a function and run it on all of Indiana
- Find the rows that do not join, and explain them
- Summarize every county with a loop, then with `groupby`
- Write one map per county to files and download them together

**You should already know:** notebooks 01a (cleaning a Census table) and 04a (functions and loops).

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## From one county to 92

Notebook 01a cleaned the ACS table for the 43 tracts of Tippecanoe County. The file
`acs_2024_indiana_tracts_raw.csv` is the same API request for **every tract in Indiana**: 1,696 rows, with the
same problems. Instead of repeating 01a step by step, we turn it into one function.

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

DP = "https://raw.githubusercontent.com/SpatialTurn/DataProcessing/main/data"
raw = pd.read_csv(f"{DP}/acs_2024_indiana_tracts_raw.csv", dtype={"state": str, "county": str, "tract": str})
tracts = gpd.read_file(f"{DP}/indiana_tracts_2024.geojson")
counties = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_counties.geojson")
print(len(raw), "rows in the table;", len(tracts), "tract boundaries;", len(counties), "counties")

## Step 1: The cleaning function

Every step of notebook 01a, in order, with the decisions written into the docstring.

In [ ]:
NAMES = {
    "B01003_001E": "pop", "B01003_001M": "pop_moe",
    "B19013_001E": "median_income", "B19013_001M": "median_income_moe",
    "B25064_001E": "median_rent", "B25064_001M": "median_rent_moe",
    "B08201_001E": "households", "B08201_001M": "households_moe",
    "B08201_002E": "hh_no_vehicle", "B08201_002M": "hh_no_vehicle_moe",
}

def clean_acs(raw):
    """Clean a raw ACS table from the Census API (read with state, county and tract as text).

    Builds GEOID, renames the variables, turns annotation codes (below -100,000,000) into NaN,
    and adds the share of households without a car (NaN where there are no households).
    """
    acs = raw.rename(columns=NAMES)
    acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
    acs["county_fips"] = acs["state"] + acs["county"]
    num = list(NAMES.values())
    codes = int(acs[num].lt(-100_000_000).sum().sum())
    acs[num] = acs[num].mask(acs[num] < -100_000_000)
    acs["pct_no_vehicle"] = acs["hh_no_vehicle"] / acs["households"].where(acs["households"] > 0) * 100
    print(f"clean_acs: {len(acs)} rows, {codes} annotation codes set to missing")
    return acs[["GEOID", "county_fips"] + num + ["pct_no_vehicle"]]

acs = clean_acs(raw)

The function prints what it did. That one line is a processing log: if next year's file has 900 codes instead
of 229, you will notice.

## Step 2: Join, and look at what does not match

In [ ]:
check = tracts.merge(acs, on="GEOID", how="outer", indicator=True)
print(check["_merge"].value_counts())
check.loc[check["_merge"] != "both", ["GEOID", "pop", "households"]]

Three tracts are in the table but have no boundary. Their codes end in `990000`, and they have no people: they
are the **water tracts** of Lake Michigan in Lake, LaPorte and Porter counties. The cartographic boundary file
leaves water-only tracts out. They can be dropped safely, and now you know why.

In [ ]:
gdf = tracts.merge(acs, on="GEOID")            # inner join: the 1,693 tracts with boundaries
print(len(gdf), "tracts joined")

## Step 3: Summarize one county

Write the summary for **one** county as a function first, and check it on Tippecanoe (`18157`) against what you
know from notebook 01a.

In [ ]:
def county_summary(t):
    """Summarize the tracts of one county: counts, totals and the share of households without a car."""
    return {
        "tracts": len(t),
        "pop": int(t["pop"].sum()),
        "households": int(t["households"].sum()),
        "pct_no_vehicle": round(t["hh_no_vehicle"].sum() / t["households"].sum() * 100, 1),
        "tracts_no_income": int(t["median_income"].isna().sum()),
    }

county_summary(gdf[gdf["county_fips"] == "18157"])

43 tracts, about 189,000 people, and 9.7% of households without a car: the same as notebook 01a. Notice that the
share is computed from the **sums** (all households without a car divided by all households), not as the average
of 43 tract percentages, which would give every tract the same weight whatever its size.

## Step 4: Loop over all 92 counties

In [ ]:
rows = []
for fips in sorted(gdf["county_fips"].unique()):
    summary = county_summary(gdf[gdf["county_fips"] == fips])
    summary["fips"] = fips
    rows.append(summary)
by_county = pd.DataFrame(rows).set_index("fips")
assert len(by_county) == 92, "expected 92 Indiana counties"
by_county.sort_values("pct_no_vehicle", ascending=False).head()

The top three are not cities: LaGrange (18087), Daviess (18027) and Adams (18001) are rural counties with large
Amish communities, many of whom do not drive. Tippecanoe, with its students, is fifth; Marion County
(Indianapolis) is tenth. A summary table raises questions as well as answering them: here it shows that "no car"
means very different things in different places.

The loop is the same pattern as in notebook 04a. pandas also has a shortcut for "split by a column, apply a
function, combine the results", `groupby().apply()`, which gives the same table:

In [ ]:
same = gdf.groupby("county_fips").apply(lambda t: pd.Series(county_summary(t)), include_groups=False)
print("same result as the loop:", (same["pop"] == by_county["pop"]).all())

## Step 5: One map per county, saved to files

Loops can also make files. This function draws one county's tracts; the loop saves a map for each county in a
list into a folder, and the last line packs the folder into a single zip file to download from the file browser
on the left.

In [ ]:
def county_map(fips, column="pct_no_vehicle", folder="maps"):
    """Save a choropleth map of one county's tracts as a PNG file, and return the file name."""
    t = gdf[gdf["county_fips"] == fips].to_crs(epsg=26916)
    name = counties.loc[counties["fips"] == fips, "name"].iloc[0]
    ax = t.plot(column=column, cmap="Purples", scheme="UserDefined", classification_kwds={"bins": [5, 10, 20, 100]},
                legend=True, edgecolor="#969696", linewidth=0.4, figsize=(6, 6),
                missing_kwds={"color": "lightgray", "hatch": "///", "label": "No households"},
                legend_kwds={"title": "Households without\na car (%)",
                             "labels": ["under 5", "5 to 10", "10 to 20", "20 or more"],
                             "loc": "upper left", "bbox_to_anchor": (1, 1)})
    ax.set_title(f"{name} County: households without a car, 2020 to 2024")
    ax.set_axis_off()
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{fips}_{name.replace(' ', '_')}.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()                        # do not keep every figure in memory
    return path

for fips in ["18157", "18097", "18105", "18089"]:     # Tippecanoe, Marion, Monroe, Lake
    try:
        print("saved", county_map(fips))
    except Exception as e:             # one bad county should not stop the others
        print("FAILED", fips, e)

shutil.make_archive("county_maps", "zip", "maps")
print(sorted(os.listdir("maps")))

Change the list to `by_county.index` and the same loop draws all 92 maps. `try` and `except` record a failure and
move on to the next county, so one problem does not cost you the other 91 maps; read the messages afterwards.

Look at one of the maps:

In [ ]:
from IPython.display import Image
Image("maps/18157_Tippecanoe.png", width=420)

## Try it yourself

Add the share of tracts **without a median income estimate** to the summary as a percentage, and list the five
counties where that share is highest. Which counties have no missing estimates at all?

*Hint:* Add a column to `by_county` from two of its columns, then sort.

<details>
<summary><b>Show a solution</b></summary>

```python
by_county["pct_tracts_no_income"] = (by_county["tracts_no_income"] / by_county["tracts"] * 100).round(1)
print(by_county.sort_values("pct_tracts_no_income", ascending=False)["pct_tracts_no_income"].head())
print((by_county["tracts_no_income"] == 0).sum(), "counties have an estimate for every tract")
```

</details>

In [ ]:
# Write your answer here

## Recap

- A cleaning function applies the same, documented steps to any size of table, and reports what it changed
- Investigate the rows that do not join: here, the three water tracts of Lake Michigan
- Summarize with sums, not averages of percentages; check the function on a county you know
- A loop or `groupby().apply()` runs the summary for all 92 counties; `assert` checks the count
- Loops can write files; `try`/`except` keeps one failure from stopping the rest

**Next:** the Data Analysis course's advanced episodes, or the Case Studies course's park access case study, which
uses the cleaning and geoprocessing steps from this module on one question.